# Try Getting Pseudobulk Diff Ex Data

In [1]:
from datasets import load_dataset
from scipy.sparse import csr_matrix
import anndata
import pandas as pd
import pubchempy as pcp

/Users/aniruddh/miniforge3/envs/distillmd312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tahoe_100m_pseudo_bulk = load_dataset('vevotx/Tahoe-100M', name='pseudobulk_differential_expression' ,streaming=True, split='train')

In [5]:
n = 5
entry = next(iter(tahoe_100m_pseudo_bulk.skip(n)))
entry


{'gene_name': 'FGR',
 'baseMean': 0.9521250128746033,
 'log2FoldChange': 3.0559072494506836,
 'lfcSE': 1.8572887182235718,
 'stat': 1.6453592777252197,
 'pvalue': 0.09989573806524277,
 'padj': None,
 'plate': '1',
 'n_cells_trt': 1378,
 'n_cells_ctrl': 4862,
 'Cell_ID_Cellosaur': 'CVCL_0023',
 'Cell_ID_DepMap': 'ACH-000681',
 'drug': '4EGI-1',
 'concentration': 0.05000000074505806,
 'concentration_unit': 'uM',
 'Cell_Name_Vevo': 'A549'}

In [3]:
from tahoe_gene_filters import (
    load_gene_metadata_frame,
    resolve_gene_biotypes,
    protein_coding_gene_symbols,
    filter_streaming_de_by_gene_name,
    save_gene_list,
    save_rows_to_jsonl,
)


In [4]:
with open("outputs/protein_coding_genes.txt", "r") as f:
    protein_coding_genes = [line.strip() for line in f if line.strip()]


In [5]:
protein_coding_de = filter_streaming_de_by_gene_name(
    tahoe_100m_pseudo_bulk,
    protein_coding_genes,
)

next(iter(protein_coding_de))


{'gene_name': 'TSPAN6',
 'baseMean': 16.41559410095215,
 'log2FoldChange': -0.27955949306488037,
 'lfcSE': 0.3667627274990082,
 'stat': -0.7622353434562683,
 'pvalue': 0.44591957330703735,
 'padj': 0.7569714784622192,
 'plate': '1',
 'n_cells_trt': 1378,
 'n_cells_ctrl': 4862,
 'Cell_ID_Cellosaur': 'CVCL_0023',
 'Cell_ID_DepMap': 'ACH-000681',
 'drug': '4EGI-1',
 'concentration': 0.05000000074505806,
 'concentration_unit': 'uM',
 'Cell_Name_Vevo': 'A549'}

In [ ]:
from tahoe_gene_filters import (
    filter_streaming_de_by_gene_name,
    save_rows_to_parquet_shards,
)

protein_coding_de = filter_streaming_de_by_gene_name(
    tahoe_100m_pseudo_bulk,
    protein_coding_genes,
)

stats = save_rows_to_parquet_shards(
    protein_coding_de,
    "outputs/protein_coding_pseudobulk_de_parquet",
    rows_per_file=750_000,
    num_workers=4,
    compression="zstd",
    prefix="part",
)

stats


Wrote 100 rows
